In [ ]:
!sudo update-alternatives --config python3


There are 2 choices for the alternative python3 (providing /usr/bin/python3).

  Selection    Path                Priority   Status
------------------------------------------------------------
* 0            /usr/bin/python3.7   2         auto mode
  1            /usr/bin/python3.6   1         manual mode
  2            /usr/bin/python3.7   2         manual mode

Press <enter> to keep the current choice[*], or type selection number: 0


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from sklearn.metrics import classification_report
import time


import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
from transformers import pipeline

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Moving 0 files to the new cache system


0it [00:00, ?it/s]

In [ ]:
from google.colab import drive


In [ ]:
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#Toys Dataset:

file = 'drive/My Drive/amazon_reviews_us_Toys_v1_00.tsv'
df=pd.read_csv(file, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [ ]:
from transformers import pipeline


In [ ]:
#Model 12 (siebert) Helper Functions

In [ ]:
def sentiment_classify(df_sample, column_name):
    sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

    for i in range (0, len(df_sample[column_name])):

        text=df_sample[column_name][i]                 
        
        if len(text) > 514:
            text = text[:514]
            
        prediction={}
        prediction = sentiment_analysis(text)

        if prediction[0]['label']=='POSITIVE':
            df_sample.loc[i, ("sentiment_analysis")]=5

        elif prediction[0]['label']== 'NEGATIVE':
            df_sample.loc[i, ("sentiment_analysis")]=1
        else: 
            print("Error.")
            
      #  if i%1000==0:
      #     print ("\n sentiment_classify:   We are at i=", str(i))    
            
            
    return df_sample
            

In [ ]:
#Model 11 / Ref cell 43 at https://github.com/sophiej-s/MSThesis-I/blob/main/wk15.ipynb
#Model 11  Helper Functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix





[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from transformers import pipeline
def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)


    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


       # if i%1000==0:
       #    print ("\n emotion_roberta:   We are at i=", str(i))    
            
            
    return df_sample

In [ ]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
      # if i%10000==0:
      #     print ("\n text_process:   We are at i=", str(i))
       
    return reviews

In [ ]:

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']



Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 

In [ ]:
def run_SVC(input_df,input_y):
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


    random_state=random.randint(0, 10000)
    print("SVC random state",random_state )

    X_train, X_test, y_train, y_test = train_test_split(input_df, input_y['star_rating'], test_size=0.33, random_state=random_state)
    
    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}

    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)
    
    return y_pred,y_test

    #clf.score(X_test, y_test)
    #y_test.value_counts()
   # print(clf.score(X_test, y_test))

#    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

#    print(confusion_matrix(y_test, y_pred))

In [ ]:
 
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))
    
    return embed_only

In [ ]:
class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])

def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics

In [ ]:
import gensim
file_embeddings_fast = 'drive/My Drive/crawl-300d-2M.vec'
word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)

/usr/local/lib/python3.7/dist-packages/gensim/similarities/__init__.py:15: UserWarning: The gensim.similarities.levenshtein submodule is disabled, because the optional Levenshtein package <https://pypi.org/project/python-Levenshtein/> is unavailable. Install Levenhstein (e.g. `pip install python-Levenshtein`) to suppress this warning.
  warnings.warn(msg)


300


In [ ]:
#Running Designs 11 and 12 

In [ ]:
#Sampling the dataset     n_samples=1000



import random
random.seed(a=12, version=2)


for i in range(0, 7):
    
    random_state=random.randint(0, 6700)
    print("random_state: ",random_state) #print random_state for reproducibility

    n_samples=1000

    
    N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=random_state)
    N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=random_state)

    samplesize=n_samples*2
    N_rewiews=N_rewiews.append(N_rewiew2)

    N_rewiews=N_rewiews.reset_index()
    N_rewiews['star_rating'].value_counts()

    #===================================
    #Running Model 11

    t_START = time.time()
    N_rewiews=text_process2(N_rewiews,'review_headline')

    emotion_roberta(N_rewiews, 'review_body')

    emotion_roberta(N_rewiews, 'review_headline')

    emotion_roberta(N_rewiews, 'review_headline_processed')
    df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 

    embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
    embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")

    embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')

    #combine the embeddings with the emotions
    embed_emptions_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')


    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
    y_pred, y_test=run_SVC(embed_emptions_combined,N_rewiews)

    elapsed = time.time() - t_START
    print("Design 11 elapsed Time is:  ", elapsed)


    #Evaluation step:
    target_names = ['0 = rating of 1',  '1 = rating of 5'] 
    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))
    print ("===================================")



    #===================================
    #Running Model 12 (siebert)
    t_START = time.time()
    sentiment_classify(N_rewiews, 'review_body')
    elapsed = time.time() - t_START
    print("Design 12 elapsed Time is:  ", elapsed)

    #Evaluation step:
    y_pred= N_rewiews['sentiment_analysis']
    y_test=N_rewiews['star_rating']

    target_names = ['0 = rating of 1',  '1 = rating of 5'] 
    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))
    print ("===================================")

    #===================================







random_state:  3887


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:92: UserWarning: `return_all_scores` is now deprecated,  if want a similar funcionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  UserWarning,


SVC random state 4407
Design 11 elapsed Time is:   436.78673028945923
                 precision    recall  f1-score   support

0 = rating of 1   0.961111  0.969188  0.965132       357
1 = rating of 5   0.963333  0.953795  0.958541       303

       accuracy                       0.962121       660
      macro avg   0.962222  0.961492  0.961837       660
   weighted avg   0.962131  0.962121  0.962106       660



Downloading:   0%|          | 0.00/687 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

...

SVC random state 6139
Design 11 elapsed Time is:   439.86222434043884
                 precision    recall  f1-score   support

0 = rating of 1   0.952941  0.923077  0.937771       351
1 = rating of 5   0.915625  0.948220  0.931638       309

       accuracy                       0.934848       660
      macro avg   0.934283  0.935648  0.934704       660
   weighted avg   0.935470  0.934848  0.934900       660

Design 12 elapsed Time is:   1708.6900556087494
                 precision    recall  f1-score   support

0 = rating of 1   0.975928  0.973000  0.974462      1000
1 = rating of 5   0.973081  0.976000  0.974538      1000

       accuracy                       0.974500      2000
      macro avg   0.974504  0.974500  0.974500      2000
   weighted avg   0.974504  0.974500  0.974500      2000

random_state:  3952


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:92: UserWarning: `return_all_scores` is now deprecated,  if want a similar funcionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  UserWarning,


SVC random state 4490
Design 11 elapsed Time is:   452.8776431083679
                 precision    recall  f1-score   support

0 = rating of 1   0.975460  0.911175  0.942222       349
1 = rating of 5   0.907186  0.974277  0.939535       311

       accuracy                       0.940909       660
      macro avg   0.941323  0.942726  0.940879       660
   weighted avg   0.943288  0.940909  0.940956       660

Design 12 elapsed Time is:   1745.1074237823486
                 precision    recall  f1-score   support

0 = rating of 1   0.974874  0.970000  0.972431      1000
1 = rating of 5   0.970149  0.975000  0.972569      1000

       accuracy                       0.972500      2000
      macro avg   0.972512  0.972500  0.972500      2000
   weighted avg   0.972512  0.972500  0.972500      2000

random_state:  5270


/usr/local/lib/python3.7/dist-packages/transformers/pipelines/text_classification.py:92: UserWarning: `return_all_scores` is now deprecated,  if want a similar funcionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  UserWarning,


SVC random state 7540
Design 11 elapsed Time is:   452.9378893375397
                 precision    recall  f1-score   support

0 = rating of 1   0.941691  0.933526  0.937591       346
1 = rating of 5   0.927445  0.936306  0.931854       314

       accuracy                       0.934848       660
      macro avg   0.934568  0.934916  0.934722       660
   weighted avg   0.934913  0.934848  0.934862       660

Design 12 elapsed Time is:   1717.1651141643524
                 precision    recall  f1-score   support

0 = rating of 1   0.981744  0.968000  0.974824      1000
1 = rating of 5   0.968442  0.982000  0.975174      1000

       accuracy                       0.975000      2000
      macro avg   0.975093  0.975000  0.974999      2000
   weighted avg   0.975093  0.975000  0.974999      2000



In [ ]:

#print out a sample review for Toys
N_rewiews['review_body'][4]



"Really cheap plastic.  It arrived broken and I didn't even expect it to last for one day as my kid's halloween costume so I returned it."